# Accelerometer-derived physical activity, sedentary time, and rest-activity rhythms

Reads the NHANES 2011-2014 minute-level wrist accelerometer files (PAXMIN), computes per-participant daily-averaged MVPA and sedentary minutes (Module A) and non-parametric rest-activity rhythm metrics (Module B), and merges them with the questionnaire-and-biomarker master file into the analytic dataset. All numerics are in `scripts/rar_core.py`; `scripts/_validate_rar_core.py` checks them against analytic answers before the minute files are read.

## Inputs

- `data/raw/PAXMIN_G.xpt`, `data/raw/PAXMIN_H.xpt`: minute-level MIMS accelerometer data, NHANES 2011-2012 and 2013-2014
- `data/processed/5_paday_dp_sl_cf_paq.csv`: questionnaire master, the Module A merge target
- `data/processed/5_paday_dp_sl_cf_paq_bio.csv`: participant master with questionnaires and biomarkers

## Outputs

- `data/csv_files/PAXMIN_G.csv`, `PAXMIN_H.csv`: CSV conversion of the raw XPT files
- `data/processed/6_obj_PA.csv`: per-participant MVPA, sedentary time, valid days, wear time and peak 30-min MIMS
- `data/processed/7_Circadian_Metrics.csv`: per-participant IS, IV, M10, L5, RA, TOTAL_MIMS and clock minutes observed
- `data/processed/7_hourly_by_day.npz`: the hourly grid and minute profiles from which the metrics are computed
- `data/processed/8_nhanes_analytic.csv`: analytic dataset, one row per participant

## Definitions

- A minute is valid when PAXQFM = 0, PAXPREDM ∈ {1, 2} (wake or sleep wear) and PAXMTSM ≥ 0 (−0.01 means MIMS was not computable). Invalid minutes are missing, not zero.
- Sedentary: MIMS < 1.05. MVPA: MIMS ≥ 19.614, the Montoye et al. (2020) non-dominant-wrist cut-point (ActiGraph AC = 3940) mapped to MIMS by Karas et al. (2022).
- A valid day has ≥ 600 valid minutes; a participant is retained with ≥ 3 valid days. Module A values are means over valid days.
- IS and IV are computed on the hourly-binned profile (Witting et al. 1990; van Someren et al. 1996). M10, L5 and RA = (M10 − L5) / (M10 + L5) are computed on the minute-level composite profile, using complete windows that wrap past midnight.

## References

Montoye AHK, Westerkamp EA, Fursa AJ, et al. (2020) Wrist-worn ActiGraph cut-points for physical activity intensity in adults.

van Someren EJ, Hagebeuk EE, Lijzenga C, et al. (1996) Circadian rest-activity rhythm disturbances in Alzheimer's disease. *Biological Psychiatry* 40:259-270.

Witting W, Kwa IH, Eikelenboom P, et al. (1990) Alterations in the circadian rest-activity rhythm in aging and Alzheimer's disease. *Biological Psychiatry* 27:563-572.

Karas M, Muschelli J, Leroux A, et al. (2022) Comparison of accelerometry-based measures of physical activity. medRxiv 2022.03.16.22272518. [AC 3940 → MIMS 19.614 on the non-dominant wrist]


## Step 0 — Imports and paths

Resolves the project root (from `NHANES_SEM_ROOT` or the working directory) and defines the input and output paths.

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import os as _os

def find_root():
    """Locate the project root from NHANES_SEM_ROOT, the working directory or its parents."""
    env = _os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE = find_root()
RAW          = BASE / 'data' / 'raw'
CSV_FILES    = BASE / 'data' / 'csv_files'
PROC         = BASE / 'data' / 'processed'

MASTER_BIO   = PROC / '5_paday_dp_sl_cf_paq_bio.csv'
OBJPA_OUT    = PROC / '6_obj_PA.csv'
RHYTHM_OUT   = PROC / '7_Circadian_Metrics.csv'
ANALYTIC_OUT = PROC / '8_nhanes_analytic.csv'

for p in (BASE, RAW, PROC):
    assert p.is_dir(), f'Missing: {p}'
CSV_FILES.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(BASE / 'scripts'))
from config import raw_file
print(f'Project root: {BASE}')
print(f'Final output: {ANALYTIC_OUT}')

## Step 1 — Convert the minute-level files to CSV, once

Reads `data/raw/PAXMIN_G` and `PAXMIN_H` (SAS transport, about 8 and 9 GB;
the name is matched without regard to case) and writes
`data/csv_files/PAXMIN_G.csv` and `PAXMIN_H.csv`. A CSV that already exists
is not converted again.

In [ ]:
for stem in ('PAXMIN_G', 'PAXMIN_H'):
    out = CSV_FILES / f'{stem}.csv'
    if out.exists():
        print(f'{out.name} exists; not converted again')
        continue
    src = raw_file(RAW, f'{stem}.xpt')
    print(f'reading {src.name} ({src.stat().st_size / 1e9:.1f} GB; several minutes)')
    df = pd.read_sas(src, format='xport')
    df.columns = [c.upper() for c in df.columns]
    df.to_csv(out, index=False)
    print(f'wrote {out.relative_to(BASE)}: {len(df):,} rows')

## Step 2 — Module A: daily-averaged MVPA and sedentary minutes

Runs `scripts/_validate_rar_core.py`, then reads `PAXMIN_G.csv` and `PAXMIN_H.csv` in chunks and summarises each person-day: valid, MVPA and sedentary minutes, and peak 30-min MIMS. Applies the ≥ 600 valid minutes per day and ≥ 3 valid days rules, averages over valid days, merges with `data/processed/5_paday_dp_sl_cf_paq.csv` and writes `data/processed/6_obj_PA.csv`.

In [ ]:
# Module A, pass 1 over the minute files. Numerics are in scripts/rar_core.py
# and are checked by scripts/_validate_rar_core.py; this cell is I/O and
# orchestration only.
# Pass 1 produces, for every (participant, day):
#     first_session_min   anchors the recording to the wall clock
#     n_valid             valid minutes -> Wear_Time and the >= 600 min/day filter
#     n_mvpa, n_sed       -> Module A person-level averages
# A minute is valid when PAXQFM == 0 and PAXPREDM in {1, 2} (wake or sleep
# wear) and PAXMTSM >= 0.
import os, sys, subprocess, importlib
import numpy as np, pandas as pd

# Paths, defined here as well as in Step 0 so that Steps 2-4 can run as a block.
from pathlib import Path
BASE         = find_root() if "find_root" in globals() else Path.cwd().parent   # set by the Step 0 cell
SCRIPTS      = str(BASE / "scripts")
CSV_DIR      = str(BASE / "data" / "csv_files")
PROC         = BASE / "data" / "processed"
SURVEY       = str(PROC / "5_paday_dp_sl_cf_paq.csv")      # Module A merge target
MASTER_BIO   = PROC / "5_paday_dp_sl_cf_paq_bio.csv"       # used by the merge cell
OBJPA_OUT    = PROC / "6_obj_PA.csv"
RHYTHM_OUT   = PROC / "7_Circadian_Metrics.csv"
ANALYTIC_OUT = PROC / "8_nhanes_analytic.csv"
OUT_A        = str(OBJPA_OUT)
FILES        = ["PAXMIN_G.csv", "PAXMIN_H.csv"]

sys.path.insert(0, SCRIPTS)
import rar_core; importlib.reload(rar_core)
from rar_core import (MIN_COLS, MVPA_THRESHOLD, SED_THRESHOLD,
                      MIN_VALID_MIN_PER_DAY, MIN_VALID_DAYS,
                      clean_chunk, summarise_chunk_pass1, combine_pass1,
                      day2_anchors, module_a_person_level,
                      top_minutes_chunk, combine_top_minutes)

# Run the ground-truth checks first
print("Running _validate_rar_core.py before reading the minute files ...")
_v = subprocess.run([sys.executable, "_validate_rar_core.py"],
                    cwd=SCRIPTS, capture_output=True, text=True)
if _v.returncode != 0:
    print(_v.stdout[-4000:])
    raise RuntimeError("rar_core validation FAILED — pipeline not started.")
print("  all ground-truth checks passed\n")

# Pass 1
parts, tops = [], []
for fn in FILES:
    path = os.path.join(CSV_DIR, fn)
    print(f"--- pass 1: {fn} ---")
    for i, chunk in enumerate(pd.read_csv(path, usecols=MIN_COLS, chunksize=1_000_000)):
        cl = clean_chunk(chunk)
        parts.append(summarise_chunk_pass1(cl))
        tops.append(top_minutes_chunk(cl))
        if i % 10 == 0:
            print(f"    chunk {i:3d}")

person_day = combine_pass1(parts)
person_day['peak30'] = combine_top_minutes(tops)
del parts, tops
print(f"\nperson-days summarised : {len(person_day):,}")

# Anchor every recording to the wall clock
anchors, inconsistent = day2_anchors(person_day)
print(f"participants anchored  : {len(anchors):,}")
print(f"anchor inconsistencies : {len(inconsistent):,}"
      + ("  <-- day starts are not on the midnight grid; check the recording" if len(inconsistent) else "  (every day d>=2 begins"
         " exactly 1440 min after day 2, as the protocol requires)"))
assert len(inconsistent) == 0, (
    "Some participants' days are not 1440 minutes apart. PAXDAYM does not "
    "reflect a gap in the recording; minute-of-day cannot be trusted.\n"
    f"{inconsistent.head(20)}")

# Inclusion and person-level averages
person, valid_pd = module_a_person_level(person_day)
print(f"\nvalid days  (>= {MIN_VALID_MIN_PER_DAY} valid min) : {len(valid_pd):,}")
print(f"participants (>= {MIN_VALID_DAYS} valid days)  : {len(person):,}")

survey  = pd.read_csv(SURVEY)
final_df = survey.merge(person, on="SEQN", how="inner")
final_df.to_csv(OUT_A, index=False)

print("\n" + "-" * 62)
print("OBJECTIVE PA SUMMARY (average daily minutes, genuine wear only)")
print(f"  merged with survey (N)   : {len(final_df):,}")
print(f"  valid days               : {final_df['Valid_Days'].mean():.1f}")
print(f"  wear time                : {final_df['Wear_Time'].mean():.1f} min/day")
print(f"  MVPA (MIMS >= {MVPA_THRESHOLD})     : {final_df['obj_mvpa_min'].mean():.1f} min/day")
print(f"  sedentary (MIMS < {SED_THRESHOLD})  : {final_df['obj_sed_min'].mean():.1f} min/day")
print("-" * 62)
print("Wear time counts valid minutes only (PAXQFM == 0, PAXPREDM in {1, 2}, PAXMTSM >= 0);")
print(f"non-wear minutes are excluded before the >= {MIN_VALID_MIN_PER_DAY} min/day filter.")
print(f"MIMS has no consensus MVPA cut-point; minutes at or above {MVPA_THRESHOLD} MIMS are")
print("reported descriptively and enter no model. Peak-30-minute MIMS below needs no cut-point.")
print("-" * 62)
if "peak30_mims" in final_df.columns:
    p = final_df["peak30_mims"].dropna()
    print(f"  peak 30-min MIMS         : {p.mean():.1f} (SD {p.std():.1f}), n = {len(p):,}")
    print( "     reference, older adults: 36.9 (SD 8.5)   Choudhury/Thiamwong 2024")
    print( "     Zheng et al. 2023 define this as the mean of the 30 highest")
    print( "     minute values in a day - an intensity summary without a threshold.")
    print("-" * 62)


In [ ]:
print(f"Average Valid Days: {final_df['Valid_Days'].mean():.1f} days")
print("objective MVPA minutes per person:")
print(final_df['obj_mvpa_min'].describe())
print("objective Sedentary minutes per person:")
print(final_df['obj_sed_min'].describe())

## Step 3 — Module B: non-parametric rest-activity rhythm metrics

Re-reads the minute files for the person-days retained in Step 2, accumulates activity by minute of day, and computes IS, IV, M10, L5, RA and total daily MIMS volume (TOTAL_MIMS) per participant. Writes `data/processed/7_Circadian_Metrics.csv` and `data/processed/7_hourly_by_day.npz`.

In [ ]:
# Module B, pass 2: clock-aligned rest-activity rhythm metrics.
#   IV   intradaily variability: fragmentation within the day
#   IS   interdaily stability: day-to-day regularity of the rhythm
#   M10  mean activity in the most active 10-h window
#   L5   mean activity in the least active 5-h window
#   RA   relative amplitude, (M10 - L5) / (M10 + L5)
# Pass 2 re-reads the minute files and accumulates activity by minute of day
# for the person-days retained in pass 1. Requires `anchors` and `valid_pd`
# from the Module A cell.
from rar_core import (ProfileAccumulator, hourly_bins, interdaily_stability,
                      intradaily_variability, m10_l5_ra)

OUT_B = str(BASE / "data" / "processed" / "7_Circadian_Metrics.csv")

valid_pd_list = list(valid_pd)
acc = ProfileAccumulator(person["SEQN"].to_numpy(), valid_pd_list)

for fn in FILES:
    path = os.path.join(CSV_DIR, fn)
    print(f"--- pass 2: {fn} ---")
    for i, chunk in enumerate(pd.read_csv(path, usecols=MIN_COLS, chunksize=1_000_000)):
        acc.add(clean_chunk(chunk), anchors)
        if i % 10 == 0:
            print(f"    chunk {i:3d}")

profiles = acc.minute_profiles()          # [n_person x 1440], NaN where unobserved
hbd      = acc.hourly_by_day()            # [n_person_day x 24]
seqns    = np.array(sorted(acc.seqn_ix))

# person -> row indices of that person's valid days
rows_by_person = {}
for i, (s, d) in enumerate(valid_pd_list):
    rows_by_person.setdefault(s, []).append(i)

print("\ncomputing IS, IV, M10, L5, RA ...")
recs = []
for s in seqns:
    prof = profiles[acc.seqn_ix[s]]
    M10, L5, RA = m10_l5_ra(prof)
    recs.append({
        "SEQN": s,
        "IS":  interdaily_stability(hbd[rows_by_person[s]]),
        "IV":  intradaily_variability(hourly_bins(prof)),
        "M10": M10, "L5": L5, "RA": RA,
        "clock_minutes_observed": int(np.isfinite(prof).sum()),
    })
# Total daily activity volume (TOTAL_MIMS): per person-day, the mean over the
# observed hours of the hourly grid scaled to 24 h; then the mean over the
# participant's valid days. Used as a covariate that separates activity volume
# from its organisation across the day. M10 is not used for this because it is
# part of RA by definition.
with np.errstate(invalid='ignore'):
    _vol_day = np.nanmean(hbd, axis=1) * 24 * 60.0
    _n_hours = np.sum(np.isfinite(hbd), axis=1)
_vol_day[_n_hours == 0] = np.nan
_vol = (pd.DataFrame({'SEQN': [s for s, d in valid_pd_list], 'v': _vol_day})
          .dropna().groupby('SEQN')['v'].mean())
circadian_df = pd.DataFrame(recs)
circadian_df['TOTAL_MIMS'] = circadian_df['SEQN'].map(_vol)
print(f'total daily MIMS volume: n = {circadian_df["TOTAL_MIMS"].notna().sum():,}  '
      f'mean = {circadian_df["TOTAL_MIMS"].mean():,.0f}')

circadian_df.to_csv(OUT_B, index=False)

# Save the hourly grid and minute profiles; every metric is a function of these
# arrays and can be recomputed without re-reading PAXMIN.
np.savez_compressed(
    str(PROC / "7_hourly_by_day.npz"),
    hourly=hbd.astype("float32"),
    seqn=np.array([s for s, d in valid_pd_list], dtype="int64"),
    day=np.array([d for s, d in valid_pd_list], dtype="int16"),
    profiles=profiles.astype("float32"),
    profile_seqn=seqns.astype("int64"))
print(f'wrote {PROC / "7_hourly_by_day.npz"} '
      f'(hourly grid retained so metrics can be recomputed without re-reading PAXMIN)')

# Diagnostics
n = len(circadian_df)
print("\n" + "-" * 70)
print(f"REST-ACTIVITY RHYTHM METRICS   N = {n:,}")
print("-" * 70)
for col, label, ref in [
    ("IV", "Intradaily variability", "0 = smooth, 2.09 = white noise at 24 bins, 4 = max"),
    ("IS", "Interdaily stability",   "1 = identical days, ~1/n_days = independent"),
    ("M10", "Most active 10 h",      "MIMS"),
    ("L5",  "Least active 5 h",      "MIMS"),
    ("RA",  "Relative amplitude",    "0-1"),
]:
    v = circadian_df[col]
    print(f"  {label:24s} mean={v.mean():8.4f}  sd={v.std():8.4f}  "
          f"median={v.median():8.4f}  n={v.notna().sum():,}")
    print(f"  {'':24s} {ref}")

print("\n  clock coverage")
cov = circadian_df["clock_minutes_observed"]
print(f"    minutes of the 1440-minute clock observed: "
      f"median={cov.median():.0f}  min={cov.min():.0f}")

print("\n  Distribution checks")
n_l5 = int((circadian_df["L5"] <= 1e-9).sum())
n_ra = int((circadian_df["RA"] >= 0.9999).sum())
print(f"    L5 exactly zero       : {n_l5:5d} / {n} ({100*n_l5/n:5.1f}%)")
print(f"    RA at ceiling (1.000) : {n_ra:5d} / {n} ({100*n_ra/n:5.1f}%)")
print(f"    IS mean               : {circadian_df['IS'].mean():.3f}   (published adult range 0.4-0.7)")
if n_ra / n > 0.25 or circadian_df["IS"].mean() < 0.25:
    print("\n    Implausible distribution (RA ceiling > 25% or mean IS < 0.25): check the wear")
    print("    classification before running 5_5_compute_kdm_ba.R and script 6.")
else:
    print("\n    Distributions are in the expected range. Proceed to 5_5_compute_kdm_ba.R, then script 6.")
print("-" * 70)


## Step 4 — Merge into the analytic dataset

Recomputes TOTAL_MIMS from `7_hourly_by_day.npz`, then left-joins the Module A output (`6_obj_PA.csv`: objective MVPA, sedentary time, wear time, valid days, peak 30-min MIMS) and the Module B output (`7_Circadian_Metrics.csv`) onto the master file `5_paday_dp_sl_cf_paq_bio.csv` on `SEQN`. Every participant in the master is kept; participants without valid accelerometer data receive missing values in the accelerometer-derived columns. Adds the pooled two-cycle weight WTMEC4YR = WTMEC2YR / 2 and writes `data/processed/8_nhanes_analytic.csv`.

In [ ]:
# TOTAL_MIMS is recomputed from the saved hourly grid (7_hourly_by_day.npz)
# each time this cell runs: mean over observed hours per person-day, scaled to
# 24 h, then averaged over valid days.
import numpy as _np
_rh = pd.read_csv(RHYTHM_OUT)
_npz = PROC / '7_hourly_by_day.npz'
if _npz.exists():
    # Recompute rather than use the stored column.
    _z = _np.load(_npz)
    _h = _z['hourly']
    with _np.errstate(invalid='ignore'):
        _v = _np.nanmean(_h, axis=1) * 24 * 60.0
        _v[_np.sum(_np.isfinite(_h), axis=1) == 0] = _np.nan
    _per = (pd.DataFrame({'SEQN': _z['seqn'], 'v': _v})
              .dropna().groupby('SEQN')['v'].mean())
    _rh['TOTAL_MIMS'] = _rh['SEQN'].map(_per)
    _rh.to_csv(RHYTHM_OUT, index=False)
    print(f'TOTAL_MIMS computed from the hourly grid for '
          f'{_rh["TOTAL_MIMS"].notna().sum():,} participants')
elif 'TOTAL_MIMS' not in _rh.columns:
    raise FileNotFoundError('No hourly grid and no stored TOTAL_MIMS; re-run Module B.')

master = pd.read_csv(MASTER_BIO)
obj_pa = pd.read_csv(OBJPA_OUT)
rhythm = pd.read_csv(RHYTHM_OUT)
print(f'master file:               {master.shape}')
print(f'objective PA file:         {obj_pa.shape}')
print(f'circadian metrics file:    {rhythm.shape}    cols = {rhythm.columns.tolist()}')

for df in (master, obj_pa, rhythm):
    df['SEQN'] = df['SEQN'].astype(int)

# Module A output carries the master questionnaire columns as well; retain only
# the new accelerometer-derived measures to avoid duplicate columns in the join.
PA_NEW = ['Valid_Days', 'Wear_Time', 'obj_mvpa_min', 'obj_sed_min', 'peak30_mims']
# clock_minutes_observed is carried through so that sensitivity to incomplete
# clock coverage can be checked downstream.
RHYTHM_NEW = ['IS', 'IV', 'M10', 'L5', 'RA', 'TOTAL_MIMS', 'clock_minutes_observed']

# Warn about upstream columns that are not in the merge lists and would be dropped.
_carried = set(master.columns) | {'SEQN'}
for _src, _keep, _name in [(obj_pa, PA_NEW, 'obj_pa'), (rhythm, RHYTHM_NEW, 'rhythm')]:
    _new = [c for c in _src.columns if c not in _carried and c not in _keep]
    if _new:
        print(f'NOTE: {_name} has columns not in the merge list and they will be '
              f'dropped: {_new}')

obj_pa_slim = obj_pa[['SEQN'] + [c for c in PA_NEW if c in obj_pa.columns]]
rhythm_slim = rhythm[['SEQN'] + [c for c in RHYTHM_NEW if c in rhythm.columns]]

analytic = master.merge(obj_pa_slim, on='SEQN', how='left').merge(rhythm_slim, on='SEQN', how='left')

# Pooled two-cycle MEC weight, stored once for all downstream scripts.
analytic['WTMEC4YR'] = analytic['WTMEC2YR'] / 2.0

collision = [c for c in analytic.columns if c.endswith('_x') or c.endswith('_y')]
assert not collision, f'Duplicate columns detected after merge: {collision}'

analytic.to_csv(ANALYTIC_OUT, index=False)
added = [c for c in analytic.columns if c not in master.columns]
print()
print(f'Wrote {ANALYTIC_OUT}')
print(f'  shape : {analytic.shape}')
print(f'  size  : {ANALYTIC_OUT.stat().st_size / 1e6:.1f} MB')
print(f'  accelerometer-derived columns added ({len(added)}): {added}')


## Step 5 — Integrity checks on the analytic dataset

Checks that `SEQN` is unique, that every column the downstream scripts require is present, that WTMEC4YR = WTMEC2YR / 2 and that the rest-activity metrics meet their acceptance criteria; reports adult sample sizes, biomarker missingness and PHQ-9 completeness.

In [ ]:
# Integrity checks on the analytic dataset before 5_5_compute_kdm_ba.R.

REQUIRED = ['IV', 'IS', 'RA', 'M10', 'L5', 'TOTAL_MIMS', 'peak30_mims',
            'clock_minutes_observed', 'obj_mvpa_min', 'obj_sed_min',
            'Wear_Time', 'Valid_Days', 'WTMEC4YR', 'SDMVSTRA', 'SDMVPSU']
_missing = [c for c in REQUIRED if c not in analytic.columns]
assert not _missing, (
    f'analytic file is missing {_missing}. Every downstream script expects '
    f'these; a merge whitelist has probably gone stale.')

assert analytic['SEQN'].is_unique
assert np.allclose(analytic['WTMEC4YR'], analytic['WTMEC2YR'] / 2.0, equal_nan=True), \
    'WTMEC4YR is not WTMEC2YR/2 - the pooled weight has been altered somewhere'

adults = analytic[analytic['RIDAGEYR'] >= 18].copy()
acc = adults.dropna(subset=['IV'])
print(f'SEQN unique {analytic["SEQN"].nunique():,} | adults {len(adults):,} | '
      f'with accelerometry {len(acc):,}')

# Acceptance criteria for the rest-activity metrics, re-checked after the merge
checks = [
    ('IS mean in 0.4-0.7',        0.40 <= acc.IS.mean() <= 0.70,   f'{acc.IS.mean():.3f}'),
    ('IV mean in 0.2-0.8',        0.20 <= acc.IV.mean() <= 0.80,   f'{acc.IV.mean():.3f}'),
    ('L5 == 0 under 5%',          (acc.L5 <= 1e-9).mean() < 0.05,  f'{100*(acc.L5<=1e-9).mean():.2f}%'),
    ('RA at ceiling under 5%',    (acc.RA >= 0.9999).mean() < 0.05, f'{100*(acc.RA>=0.9999).mean():.2f}%'),
    # NaN fails between() and >, so compare only where the value exists. RA and
    # M10 are missing when the profile has no complete 600-minute window.
    ('RA within [0,1]',           acc.RA.dropna().between(0, 1).all(),
                                  f'{acc.RA.min():.3f}-{acc.RA.max():.3f}, '
                                  f'{acc.RA.isna().sum()} missing'),
    ('M10 > L5 where both exist', (acc.dropna(subset=['M10', 'L5'])
                                   .eval('M10 > L5')).all(),        ''),
    ('IS within [0,1]',           acc.IS.dropna().between(0, 1).all(),
                                  f'{acc.IS.min():.3f}-{acc.IS.max():.3f}'),
    ('clock coverage median 1440', acc.clock_minutes_observed.median() == 1440, ''),
]
print()
for label, ok, val in checks:
    print(f'  [{"ok" if ok else "FAIL"}] {label:28s} {val}')
assert all(ok for _, ok, _ in checks), \
    'A rest-activity metric fails its acceptance criterion after the merge.'

print('\nRest-activity rhythm metrics, adults with accelerometry:')
print(acc[['IV', 'IS', 'RA', 'M10', 'L5']].describe().round(3).to_string())

print('\nActivity, adults with accelerometry:')
print(acc[['peak30_mims', 'TOTAL_MIMS', 'obj_mvpa_min', 'obj_sed_min',
           'Wear_Time', 'Valid_Days']].describe().round(2).to_string())
print(f'\n  peak 30-min MIMS by age: '
      + '  '.join(f'{lab} {acc[m].peak30_mims.mean():.1f}' for lab, m in
                  [('18-39', acc.RIDAGEYR < 40),
                   ('40-59', (acc.RIDAGEYR >= 40) & (acc.RIDAGEYR < 60)),
                   ('60+',   acc.RIDAGEYR >= 60)]))
print('  reference for 60+: 36.9 (SD 8.5), community-dwelling older adults')

print('\nBiomarker missingness, adults:')
for c in ['SII', 'NLR', 'MLR', 'LBXSAL', 'LBXSCR', 'LBXGH', 'LBXTC', 'LBXSGL']:
    if c in adults.columns:
        print(f'  {c:10s} n = {int(adults[c].notna().sum()):5,}   '
              f'missing = {adults[c].isna().mean()*100:5.1f}%')

# PHQ-9 is DPQ010..DPQ090; DPQ100 (functional difficulty) is not part of the score.
DPQ = [f'DPQ{i:03d}' for i in range(10, 100, 10)]
have = [c for c in DPQ if c in adults.columns]
assert len(have) == 9, f'expected 9 PHQ-9 items, found {len(have)}: {have}'
print(f'\nPHQ-9 complete (all nine items): n = '
      f'{int(adults[have].notna().all(axis=1).sum()):,}')

nxt = 'Rscript scripts/5_5_compute_kdm_ba.R' if 'KDM_BA' not in analytic.columns \
      else '9_mplus_prep'
print(f'\nAll checks passed. Next: {nxt}')
